# Patent similarity: a controlled context study
**Name: Priyanshi Goyal · Roll number: A24ARIU0009**

This notebook reorganizes the course problem as an experiment registry: a frozen sentence encoder,
a section-context regression reference, and two single-factor context ablations. Each run writes its
configuration, elapsed training time, validation predictions and test predictions. Model selection
uses validation only. This is a newly structured implementation informed by the
[course starter and brief](https://github.com/bk-anupam/generative_ai_shared/tree/master/module3/labs/patent_phrase_matching),
which remain the source of the task, reference settings and CPC section descriptions.

**Execution status:** no training or leaderboard scores are claimed in this file. Run on a GPU with
competition data and local checkpoints attached. The reference configuration matches the starter;
Task 1's separately required unchanged-starter run must still be recorded by the student.


## Reproducible inputs
Install `torch transformers datasets accelerate scikit-learn pandas sentence-transformers sentencepiece`
in an online preparation session. Attach local DeBERTa-v3-small and all-MiniLM-L6-v2 directories for an
offline Kaggle run. Set `PATENT_DATA`, `PATENT_MODEL` and `PATENT_EMBEDDINGS` if automatic discovery
is unsuitable. No test labels or external prediction APIs are used.


In [ ]:
import os, json, time, random, gc, math
from pathlib import Path
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, Trainer, TrainingArguments
SEED = 42
transformers.set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT_OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print({'student': 'Priyanshi Goyal', 'roll': 'A24ARIU0009',
       'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})


In [ ]:
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from tokenizers.processors import TemplateProcessing
from sentence_transformers import SentenceTransformer
OUT = ROOT_OUT / 'priyanshi_patent_A24ARIU0009'
OUT.mkdir(parents=True, exist_ok=True)

def local_directory(env, suffix):
    explicit = os.environ.get(env)
    if explicit:
        path = Path(explicit)
        if not path.is_dir(): raise FileNotFoundError(path)
        return path
    hits = list(Path('/kaggle/input').glob('**/' + suffix))
    if len(hits) != 1:
        raise ValueError(f'Set {env} to one local directory; found {len(hits)} matches for {suffix}')
    return hits[0].parent

DATA = local_directory('PATENT_DATA', 'us-patent-phrase-to-phrase-matching/train.csv')
MODEL = local_directory('PATENT_MODEL', 'deberta-v3-small/config.json')
EMBEDDINGS = local_directory('PATENT_EMBEDDINGS', 'all-MiniLM-L6-v2/config.json')
frame, test = pd.read_csv(DATA/'train.csv'), pd.read_csv(DATA/'test.csv')
required = {'id', 'anchor', 'target', 'context'}
assert required <= set(frame) and required <= set(test) and 'score' in frame
assert not frame[list(required)+['score']].isna().any().any()
assert not test[list(required)].isna().any().any()
assert frame.id.is_unique and test.id.is_unique
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
fit_ids, hold_ids = next(split.split(frame, groups=frame.anchor))
fit, hold = frame.iloc[fit_ids].copy(), frame.iloc[hold_ids].copy()
assert set(fit.anchor).isdisjoint(hold.anchor)
frame.assign(partition=np.where(frame.index.isin(hold_ids), 'validation', 'train'))[['id','partition']].to_csv(OUT/'split.csv', index=False)
print({'train_rows':len(fit), 'validation_rows':len(hold), 'test_rows':len(test)})

def pearson(y, p):
    y, p = np.asarray(y), np.asarray(p)
    if len(y)<2 or np.std(y)==0 or np.std(p)==0: return float('nan')
    return float(np.corrcoef(y,p)[0,1])


## Frozen baseline and shared validation
An anchor occurs with several targets. Holding out complete anchors prevents memorized anchor
associations from inflating validation performance. A bi-encoder embeds each phrase independently;
a cross-encoder jointly attends to both phrases. Their difference therefore reflects both supervised
adaptation and architecture, not a pure estimate of training alone.


In [ ]:
encoder = SentenceTransformer(str(EMBEDDINGS), local_files_only=True, device=DEVICE)
start = time.perf_counter()
a = encoder.encode(hold.anchor.tolist(), normalize_embeddings=True, show_progress_bar=True)
b = encoder.encode(hold.target.tolist(), normalize_embeddings=True, show_progress_bar=True)
frozen_predictions = (a*b).sum(axis=1)
records = [{'run':'frozen_biencoder', 'changed':'architecture + no supervised training',
            'pearson':pearson(hold.score, frozen_predictions), 'train_seconds':0.0,
            'inference_seconds':time.perf_counter()-start}]
hold.assign(pred=frozen_predictions).to_csv(OUT/'frozen_validation.csv',index=False)
del encoder, a, b
gc.collect()
if DEVICE=='cuda': torch.cuda.empty_cache()


In [ ]:
SECTIONS = {'A':'human necessities','B':'performing operations; transporting',
'C':'chemistry; metallurgy','D':'textiles; paper','E':'fixed constructions',
'F':'mechanical engineering; lighting; heating; weapons; blasting','G':'physics','H':'electricity'}
tokenizer = AutoTokenizer.from_pretrained(str(MODEL), local_files_only=True)
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single='[CLS] $A [SEP]', pair='[CLS] $A [SEP] $B:1 [SEP]:1',
    special_tokens=[('[CLS]',tokenizer.cls_token_id),('[SEP]',tokenizer.sep_token_id)])

EXPERIMENTS = [
    {'name':'section_reference','context':'section','changed':'reference'},
    {'name':'without_context','context':'none','changed':'remove CPC section'},
    {'name':'raw_cpc','context':'code','changed':'replace section with raw CPC code'},
]

def encoded(rows, context, labelled=True):
    right = rows.target.astype(str)
    if context=='section':
        section = rows.context.str[0].map(SECTIONS)
        assert section.notna().all(), 'Unexpected CPC section'
        right = right + '; ' + section
    elif context=='code': right = right + '; ' + rows.context
    elif context!='none': raise ValueError(context)
    result = tokenizer(rows.anchor.tolist(), right.tolist(), truncation=True, max_length=64)
    if labelled: result['labels'] = rows.score.astype('float32').tolist()
    return Dataset.from_dict(result)

def run_trial(spec):
    transformers.set_seed(SEED)  # same initial head, shuffle and seed for each trial
    directory = OUT/spec['name']
    directory.mkdir(exist_ok=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL), local_files_only=True, num_labels=1, dtype=torch.float32)
    args = TrainingArguments(output_dir=str(directory), num_train_epochs=3, learning_rate=2e-5,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        warmup_ratio=0.1, weight_decay=0.01, eval_strategy='epoch', save_strategy='no',
        logging_steps=100, fp16=DEVICE=='cuda', report_to='none', seed=SEED)
    trainer = Trainer(model=model,args=args,train_dataset=encoded(fit,spec['context']),
        eval_dataset=encoded(hold,spec['context']),data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=lambda x:{'pearson':pearson(x.label_ids,x.predictions.reshape(-1))})
    if DEVICE=='cuda': torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    trainer.train()
    if DEVICE=='cuda': torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    pred=trainer.predict(encoded(hold,spec['context'])).predictions.reshape(-1)
    test_pred=trainer.predict(encoded(test,spec['context'],False)).predictions.reshape(-1)
    assert np.isfinite(pred).all() and np.isfinite(test_pred).all()
    hold.assign(pred=pred).to_csv(directory/'validation.csv',index=False)
    pd.DataFrame({'id':test.id,'score':np.clip(test_pred,0,1)}).to_csv(directory/'submission.csv',index=False)
    (directory/'config.json').write_text(json.dumps(spec|{'seed':SEED,'epochs':3,'lr':2e-5},indent=2))
    row={'run':spec['name'],'changed':spec['changed'],'pearson':pearson(hold.score,pred),
         'train_seconds':elapsed,'peak_gpu_gb':torch.cuda.max_memory_allocated()/1e9 if DEVICE=='cuda' else None}
    del trainer, model
    gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()
    return row

for spec in EXPERIMENTS:
    records.append(run_trial(spec))
    pd.DataFrame(records).to_csv(OUT/'experiments.csv',index=False)
pd.DataFrame(records)


## Select by validation, then diagnose failures
The largest validation correlation selects the final supervised trial. Small differences (about 0.005)
need repeated seeds before a superiority claim. The saved split is identical across all trials.
Per-section correlations are undefined for constant labels or predictions and are shown as missing.


In [ ]:
scores = pd.DataFrame(records)
eligible = scores[scores.run.isin([s['name'] for s in EXPERIMENTS])].dropna(subset=['pearson'])
if eligible.empty: raise RuntimeError('No finite supervised validation correlation')
best = eligible.sort_values('pearson',ascending=False).iloc[0]['run']
errors = pd.read_csv(OUT/best/'validation.csv')
errors['section'] = errors.context.str[0]
errors['absolute_error'] = abs(errors.score-errors.pred)
section_rows = [{'section':section,'n':len(g),'pearson':pearson(g.score,g.pred),
                 'mae':g.absolute_error.mean()} for section,g in errors.groupby('section')]
pd.DataFrame(section_rows).to_csv(OUT/'section_errors.csv',index=False)
calibration = errors.groupby('score').agg(n=('id','size'),mean_prediction=('pred','mean'),mae=('absolute_error','mean'))
calibration.to_csv(OUT/'score_calibration.csv')
worst=errors.nlargest(10,'absolute_error').copy()
worst['human_failure_category']=''
worst['label_discussion']=''
worst.to_csv(OUT/'review_ten_pairs.csv',index=False)
submission=pd.read_csv(OUT/best/'submission.csv')
assert submission.id.astype(str).tolist()==test.id.astype(str).tolist()
submission.to_csv(ROOT_OUT/'submission.csv',index=False)
print('Selected:',best)
print(pd.DataFrame(section_rows).to_string(index=False))
print(calibration)
worst[['anchor','target','context','score','pred']]


## Interpretation and evidence to add after running
Inspect `score_calibration.csv` for compression at 0 and 1. Use actual examples in
`review_ten_pairs.csv` to categorize abbreviation, field-specific meaning, or lexical-overlap errors;
do not infer categories solely from error magnitude. Compare the chosen model with the frozen baseline.

For the required late submission, attach checkpoints and dependency wheels, disable internet, run
all cells, and submit the notebook on Kaggle. Record every public/private score and its screenshot
in a separate report; none has been generated here. Differences from validation may reflect unseen
anchors, context distribution and model selection on the validation set. For a faster final notebook,
freeze the selected configuration after exploration and run only that trial.

Sources: course brief/starter linked above; [Sentence-BERT](https://arxiv.org/abs/1908.10084);
[DeBERTaV3](https://arxiv.org/abs/2111.09543). The competition data must be obtained through Kaggle.
